In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier

In [4]:
df = pd.read_csv(
    "../dataset/accident_data.csv"
)

In [9]:
df.head()

,Accident_Index,1st_Road_Class,1st_Road_Number,2nd_Road_Class,2nd_Road_Number,Accident_Severity,Carriageway_Hazards,Date,Day_of_Week,Did_Police_Officer_Attend_Scene_of_Accident,...,Skidding_and_Overturning,Towing_and_Articulation,Vehicle_Leaving_Carriageway,Vehicle_Location.Restricted_Lane,Vehicle_Manoeuvre,Vehicle_Reference,Vehicle_Type,Was_Vehicle_Left_Hand_Drive,X1st_Point_of_Impact,Year
0,200754AM08507,A,345,NaN,0.0,Fatal,NaN,28-01-2021,Tuesday,1,...,Skidded,No tow/articulation,Offside,0,Overtaking moving vehicle - offside,1,Car,No,Back,2021
1,2009559D02192,A,30,NaN,0.0,Fatal,NaN,29-11-2021,Thursday,1,...,NaN,No tow/articulation,Offside,0,Going ahead left-hand bend,1,Motorcycle over 500cc,No,Front,2020
2,201054MB04210,Motorway,4,NaN,0.0,Fatal,NaN,15-03-2021,Saturday,1,...,NaN,No tow/articulation,Did not leave carriageway,0,Going ahead other,2,Motorcycle over 125cc and up to 500cc,No,Back,2019
3,201014A194610,Motorway,18,A,630.0,Fatal,NaN,08-07-2020,Tuesday,1,...,NaN,No tow/articulation,Did not leave carriageway,0,Reversing,3,Other vehicle,No,Back,2019
4,201014A194610,Motorway,18,A,630.0,Fatal,NaN,30-11-2010,Tuesday,1,...,NaN,No tow/articulation,Did not leave carriageway,0,Reversing,3,Other vehicle,No,Back,2010


In [8]:
df.dtypes
df.columns

Index(['Accident_Index', '1st_Road_Class', '1st_Road_Number', '2nd_Road_Class',
       '2nd_Road_Number', 'Accident_Severity', 'Carriageway_Hazards', 'Date',
       'Day_of_Week', 'Did_Police_Officer_Attend_Scene_of_Accident',
       'Junction_Control', 'Junction_Detail', 'Latitude', 'Light_Conditions',
       'Location_Easting_OSGR', 'Location_Northing_OSGR', 'Longitude',
       'LSOA_of_Accident_Location', 'Number_of_Casualties',
       'Number_of_Vehicles', 'Pedestrian_Crossing-Human_Control',
       'Pedestrian_Crossing-Physical_Facilities', 'Road_Surface_Conditions',
       'Road_Type', 'Special_Conditions_at_Site', 'Speed_limit', 'Time',
       'Urban_or_Rural_Area', 'Weather_Conditions', 'InScotland',
       'Age_Band_of_Driver', 'Age_of_Vehicle', 'Driver_Home_Area_Type',
       'Driver_IMD_Decile', 'Engine_Capacity_.CC.',
       'Hit_Object_in_Carriageway', 'Hit_Object_off_Carriageway',
       'Journey_Purpose_of_Driver', 'Junction_Location', 'make', 'model',
       'Propulsion

In [10]:
selected_cols = [

    "Latitude",
    "Longitude",

    "Weather_Conditions",
    "Road_Type",
    "Light_Conditions",
    "Road_Surface_Conditions",
    "Urban_or_Rural_Area",
    "Junction_Detail",

    "Speed_limit",

    "Date",
    "Time",

    "Accident_Severity"
]

df = df[selected_cols]

In [11]:
df = df.dropna()

In [12]:
df["risk"] = df["Accident_Severity"].apply(

    lambda x: 1
    if x in ["Fatal", "Serious"]
    else 0
)

In [13]:
df["Date"] = pd.to_datetime(
    df["Date"],
    dayfirst=True
)

In [14]:
df["month"] = df["Date"].dt.month

df["day"] = df["Date"].dt.day

df["weekday"] = df["Date"].dt.weekday

In [16]:
df["hour"] = pd.to_datetime(
    df["Time"],
    format="%H:%M:%S"
).dt.hour

df["is_peak_hour"] = df["hour"].apply(

    lambda x: 1
    if 7 <= x <= 10 or 17 <= x <= 20
    else 0
)

df["is_weekend"] = df["weekday"].apply(

    lambda x: 1
    if x >= 5
    else 0
)

df["is_night"] = df["hour"].apply(

    lambda x: 1
    if x >= 20 or x <= 5
    else 0
)

In [17]:
cat_cols = [

    "Weather_Conditions",
    "Road_Type",
    "Light_Conditions",
    "Road_Surface_Conditions",
    "Urban_or_Rural_Area",
    "Junction_Detail"
]

label_encoders = {}

In [20]:
for col in cat_cols:

    le = LabelEncoder()

    df[col] = le.fit_transform(
        df[col].astype(str)
    )

    label_encoders[col] = le

In [10]:
X = df.drop(

    columns=[
        "risk",
        "Accident_Severity",
        "Date",
        "Time"
    ]
)

y = df["risk"]

KeyError: "['risk'] not found in axis"

In [9]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

NameError: name 'X' is not defined

In [24]:
print(
    df["risk"].value_counts()
)

print(
    df["risk"].value_counts(normalize=True)
)

risk
0    17775
1     8226
Name: count, dtype: int64
risk
0    0.683628
1    0.316372
Name: proportion, dtype: float64


In [5]:
model = XGBClassifier(

    n_estimators=300,

    max_depth=8,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    objective='binary:logistic',

    eval_metric='logloss',

    random_state=42
)

In [8]:
model.fit(
    X_train,
    y_train
)

NameError: name 'X_train' is not defined

In [27]:
y_pred = model.predict(X_test)

In [28]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.89      0.82      0.85      3556
           1       0.67      0.77      0.71      1645

    accuracy                           0.81      5201
   macro avg       0.78      0.80      0.78      5201
weighted avg       0.82      0.81      0.81      5201



In [29]:
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

[[2921  635]
 [ 378 1267]]


In [30]:
print(
    roc_auc_score(
        y_test,
        y_pred
    )
)

0.795820668693009


In [ ]:
# joblib.dump(

#     model,

#     "../models/xgboost_roadsafe.pkl"
# )

# joblib.dump(

#     label_encoders,

#     "../models/label_encoders.pkl"
# )

# joblib.dump(

#     X.columns.tolist(),

#     "../models/feature_columns.pkl"
# )

['../models/feature_columns.pkl']

In [32]:
print(
    label_encoders["Weather_Conditions"].classes_
)

['Fine + high winds' 'Fine no high winds' 'Fog or mist' 'Other'
 'Raining + high winds' 'Raining no high winds' 'Snowing + high winds'
 'Snowing no high winds' 'Unknown']


NameError: name 'df' is not defined

In [4]:
sorted(df["Weather_Conditions"].unique())

['Fine + high winds',
 'Fine no high winds',
 'Fog or mist',
 'Other',
 'Raining + high winds',
 'Raining no high winds',
 'Snowing + high winds',
 'Snowing no high winds',
 'Unknown']

In [7]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)

NameError: name 'X_test' is not defined